# 🤟 ASL Alphabet — Full Pipeline
### Raw Images → Preprocess → YOLO Dataset → Train YOLOv8m

```
📁 Raw Dataset (3000 imgs/class)
        ↓  sample 200/class (50 × 4 ranges)
🔬 Denoise   (Non-Local Means)
        ↓
⚖️  Normalize  (CLAHE)
        ↓
🔄 Augment   (Rotate ± 15°, Brightness/Contrast, V-Flip)
        ↓
🗂️  Split     Train 75% │ Val 15% │ Test 10%
        ↓
🎯 YOLO Labels  (class x_c y_c w h)
        ↓
🚀 Train YOLOv8m
        ↓
📊 Evaluate & Export
```

---
> ⚠️ **Only change the paths in Step 2 — CONFIG.** Everything else runs automatically.

---
## 📦 Step 1 — Install Dependencies

In [1]:
import sys

# Detect CUDA & install correct PyTorch
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    import re
    match = re.search(r'CUDA Version: (\d+)', result.stdout)
    cuda_major = int(match.group(1)) if match else 0
    if cuda_major >= 12:
        whl = 'https://download.pytorch.org/whl/cu121'
    else:
        whl = 'https://download.pytorch.org/whl/cu118'
    print(f'🖥️  CUDA {cuda_major} detected → installing PyTorch...')
    !{sys.executable} -m pip install torch torchvision --index-url {whl} -q
else:
    print('⚠️  No GPU found — installing CPU PyTorch')
    !{sys.executable} -m pip install torch torchvision -q

!{sys.executable} -m pip install ultralytics opencv-python pyyaml pillow numpy matplotlib seaborn tqdm scikit-image -q
print('\n✅ All dependencies installed')

🖥️  CUDA 13 detected → installing PyTorch...
/bin/bash: line 1: /home/abdelrahman/Downloads/SUTECH/third: No such file or directory
/bin/bash: line 1: /home/abdelrahman/Downloads/SUTECH/third: No such file or directory

✅ All dependencies installed


---
## ⚙️ Step 2 — CONFIG  *(only edit this cell)*

In [ ]:
# ══════════════════════════════════════════════════════════
#  ✏️  EDIT THESE VALUES
# ══════════════════════════════════════════════════════════
DATASET_ROOT   = '/home/abdelrahman/.cache/kagglehub/datasets/grassknoted/asl-alphabet/versions/1/asl_alphabet_train/asl_alphabet_train'
PROCESSED_ROOT = './asl_processed'   # denoised + normalized + augmented
YOLO_ROOT      = './asl_yolo'        # final YOLO dataset (train/val/test)
RUNS_DIR       = './runs'            # training output

# ── Sampling ──────────────────────────────────────────────
RANGES = [
    (0,    750,  50),
    (750,  1250, 50),
    (1250, 1700, 50),
    (2000, 3000, 50),
]

# ── Splits ────────────────────────────────────────────────
TRAIN_SPLIT = 0.75   # 75%
VAL_SPLIT   = 0.15   # 15%
TEST_SPLIT  = 0.10   # 10%

# ── Preprocessing ─────────────────────────────────────────
DENOISE_METHOD = 'nlm'   # 'gaussian' | 'median' | 'nlm'
AUG_PER_IMAGE  = 3       # augmented copies per original image

# ── Training ──────────────────────────────────────────────
EPOCHS    = 50
PATIENCE  = 10
IMG_SIZE  = 200
WORKERS   = 8
# ══════════════════════════════════════════════════════════

CLASSES = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z',
    'del','nothing','space'
]

import random, os
import numpy as np
from pathlib import Path
random.seed(42)
np.random.seed(42)

assert Path(DATASET_ROOT).exists(), f'❌ Dataset not found: {DATASET_ROOT}'

print('✅ Config loaded')
print(f'   Source   : {DATASET_ROOT}')
print(f'   Splits   : train={int(TRAIN_SPLIT*100)}%  val={int(VAL_SPLIT*100)}%  test={int(TEST_SPLIT*100)}%')
print(f'   Aug/img  : {AUG_PER_IMAGE}  →  ~{200*(1+AUG_PER_IMAGE)} images/class after augmentation')

---
## 🔬 Step 3 — Denoise + Normalize Functions

In [ ]:
import cv2

# ── Denoise ───────────────────────────────────────────────
def denoise_gaussian(img):
    return cv2.GaussianBlur(img, (3, 3), 0)

def denoise_median(img):
    return cv2.medianBlur(img, 3)

def denoise_nlm(img):
    return cv2.fastNlMeansDenoisingColored(img, None,
        h=10, hColor=10, templateWindowSize=7, searchWindowSize=21)

DENOISE_FNS = {'gaussian': denoise_gaussian, 'median': denoise_median, 'nlm': denoise_nlm}

# ── Normalize (CLAHE) ─────────────────────────────────────
def normalize_image(img_bgr):
    lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    img_clahe = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return (img_clahe.astype('float32') / 255.0 * 255).astype('uint8')

# ── Augment ───────────────────────────────────────────────
def augment_image(img_bgr, seed=None):
    if seed is not None:
        np.random.seed(seed)
    h, w = img_bgr.shape[:2]
    out  = img_bgr.copy()
    # Rotate ±15°
    angle = np.random.uniform(-15, 15)
    M     = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    out   = cv2.warpAffine(out, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    # Brightness & Contrast
    out = cv2.convertScaleAbs(out, alpha=np.random.uniform(0.7,1.3), beta=np.random.uniform(-30,30))
    # Vertical flip 30%  (safe for ASL — horizontal flip changes sign meaning!)
    if np.random.rand() < 0.3:
        out = cv2.flip(out, 0)
    return out

print(f'✅ Denoise ({DENOISE_METHOD.upper()}) + Normalize (CLAHE) + Augment ready')

---
## 🖼️ Step 4 — Preview: Denoise Comparison

In [ ]:
import matplotlib.pyplot as plt

root = Path(DATASET_ROOT)
sample_path = random.choice(list((root / 'A').glob('*.jpg')))
orig = cv2.imread(str(sample_path))

samples = {
    'Original'        : orig,
    'Gaussian'        : denoise_gaussian(orig),
    'Median'          : denoise_median(orig),
    'Non-Local Means' : denoise_nlm(orig),
    'NLM + CLAHE'     : normalize_image(denoise_nlm(orig)),
    'NLM + CLAHE + Aug': augment_image(normalize_image(denoise_nlm(orig)), seed=7),
}

fig, axes = plt.subplots(1, 6, figsize=(20, 4))
fig.suptitle('Full Preprocessing Preview', fontsize=13, fontweight='bold')
for ax, (title, img) in zip(axes, samples.items()):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('preprocessing_preview.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Saved → preprocessing_preview.png')

---
## ⚙️ Step 5 — Process All Images  *(Denoise + Normalize + Augment)*

In [ ]:
import shutil
from tqdm import tqdm

def sample_from_ranges(images, ranges):
    images_sorted = sorted(images, key=lambda p: p.name)
    total    = len(images_sorted)
    selected = []
    for (start, end, n) in ranges:
        bucket = images_sorted[min(start, total):min(end, total)]
        selected.extend(random.sample(bucket, min(n, len(bucket))))
    return selected

proc_out   = Path(PROCESSED_ROOT)
denoise_fn = DENOISE_FNS[DENOISE_METHOD]
total_saved = 0

for cls_name in tqdm(CLASSES, desc='Processing'):
    cls_dir  = root / cls_name
    save_dir = proc_out / cls_name
    save_dir.mkdir(parents=True, exist_ok=True)

    all_imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    selected = sample_from_ranges(all_imgs, RANGES)

    for img_path in selected:
        img = cv2.imread(str(img_path))
        if img is None: continue

        img = denoise_fn(img)
        img = normalize_image(img)

        # Save original processed
        cv2.imwrite(str(save_dir / f'{img_path.stem}_proc.jpg'), img)
        total_saved += 1

        # Save augmented copies
        for a in range(AUG_PER_IMAGE):
            aug = augment_image(img, seed=hash(img_path.stem) + a)
            cv2.imwrite(str(save_dir / f'{img_path.stem}_aug{a}.jpg'), aug)
            total_saved += 1

print(f'\n✅ Processed {total_saved} images → {proc_out.resolve()}')

---
## 🗂️ Step 6 — Convert to YOLO Format  *(Train / Val / Test split)*

In [ ]:
import yaml

def write_label(label_path, class_id):
    label_path.parent.mkdir(parents=True, exist_ok=True)
    with open(label_path, 'w') as f:
        f.write(f'{class_id} 0.5 0.5 1.0 1.0\n')

yolo_out  = Path(YOLO_ROOT)
class2id  = {c: i for i, c in enumerate(CLASSES)}

for split in ('train', 'val', 'test'):
    (yolo_out / 'images' / split).mkdir(parents=True, exist_ok=True)
    (yolo_out / 'labels' / split).mkdir(parents=True, exist_ok=True)

total_train = total_val = total_test = 0

for cls_name in tqdm(CLASSES, desc='YOLO conversion'):
    cls_dir = proc_out / cls_name
    images  = list(cls_dir.glob('*.jpg'))
    if not images: continue

    random.shuffle(images)
    n       = len(images)
    n_test  = max(1, int(n * TEST_SPLIT))
    n_val   = max(1, int(n * VAL_SPLIT))

    splits = {
        'test' : images[:n_test],
        'val'  : images[n_test : n_test + n_val],
        'train': images[n_test + n_val:],
    }

    for split, split_imgs in splits.items():
        for img_path in split_imgs:
            dst_img = yolo_out / 'images' / split / f'{cls_name}_{img_path.name}'
            dst_lbl = yolo_out / 'labels' / split / f'{cls_name}_{img_path.stem}.txt'
            shutil.copy2(img_path, dst_img)
            write_label(dst_lbl, class2id[cls_name])

    total_train += len(splits['train'])
    total_val   += len(splits['val'])
    total_test  += len(splits['test'])

# Write data.yaml
yaml_path = yolo_out / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump({
        'path' : str(yolo_out.resolve()),
        'train': 'images/train',
        'val'  : 'images/val',
        'test' : 'images/test',
        'nc'   : len(CLASSES),
        'names': CLASSES,
    }, f, default_flow_style=False, sort_keys=False)

grand = total_train + total_val + total_test
print('\n' + '='*55)
print('  ✅ YOLO Dataset Ready!')
print(f'  Train  : {total_train:5d}  ({100*total_train//grand}%)')
print(f'  Val    : {total_val:5d}  ({100*total_val//grand}%)')
print(f'  Test   : {total_test:5d}  ({100*total_test//grand}%)')
print(f'  Total  : {grand:5d}')
print(f'  YAML   : {yaml_path.resolve()}')
print('='*55)

---
## 📊 Step 7 — Dataset Distribution Chart

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

counts = {}
colors = {'train': 'steelblue', 'val': 'coral', 'test': 'mediumseagreen'}

for split in ('train', 'val', 'test'):
    imgs = list((yolo_out / 'images' / split).glob('*.jpg'))
    counts[split] = Counter(p.name.split('_')[0] for p in imgs)

x     = list(range(len(CLASSES)))
width = 0.28

fig, ax = plt.subplots(figsize=(20, 6))
for i, (split, color) in enumerate(colors.items()):
    vals = [counts[split].get(c, 0) for c in CLASSES]
    bars = ax.bar([xi + i*width for xi in x], vals, width, label=split.capitalize(), color=color, edgecolor='white')

ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_title('Dataset Split Distribution per Class', fontsize=14, fontweight='bold')
ax.set_ylabel('Images')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('split_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Saved → split_distribution.png')

---
## 🚀 Step 8 — Train YOLOv8m

In [ ]:
import torch
from ultralytics import YOLO

# Auto batch size based on VRAM
if torch.cuda.is_available():
    vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 64 if vram >= 16 else 32 if vram >= 8 else 16 if vram >= 4 else 8
    print(f'🖥️  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram:.1f} GB  |  Batch: {BATCH}')
    DEVICE = 0
else:
    BATCH  = 8
    DEVICE = 'cpu'
    print('⚠️  No GPU — training on CPU')

model = YOLO('yolov8m.pt')

results = model.train(
    data          = str(yaml_path),
    epochs        = EPOCHS,
    patience      = PATIENCE,
    imgsz         = IMG_SIZE,
    batch         = BATCH,
    workers       = WORKERS,
    device        = DEVICE,
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,
    augment       = True,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    fliplr        = 0.0,       # disabled — flipping changes ASL meaning!
    degrees       = 10,
    translate     = 0.1,
    scale         = 0.3,
    project       = RUNS_DIR,
    name          = 'asl_yolov8m',
    exist_ok      = True,
    verbose       = True,
    plots         = True,
    save          = True,
    save_period   = 10,
)

BEST_PT = f'{RUNS_DIR}/asl_yolov8m/weights/best.pt'
print(f'\n✅ Training done!  Best weights → {BEST_PT}')

---
## 📊 Step 9 — Evaluate on Val & Test Sets

In [ ]:
best_model = YOLO(BEST_PT)

for split_name in ('val', 'test'):
    print(f'\n📈 {split_name.upper()} SET RESULTS:')
    m = best_model.val(
        data   = str(yaml_path),
        split  = split_name,
        imgsz  = IMG_SIZE,
        device = DEVICE,
    )
    print(f'   mAP50      : {m.box.map50:.4f}')
    print(f'   mAP50-95   : {m.box.map:.4f}')
    print(f'   Precision  : {m.box.mp:.4f}')
    print(f'   Recall     : {m.box.mr:.4f}')

---
## 📈 Step 10 — Plot Training Curves

In [ ]:
import matplotlib.image as mpimg

results_dir = f'{RUNS_DIR}/asl_yolov8m'
plots = [
    ('results.png',           'Loss & mAP Curves'),
    ('confusion_matrix.png',  'Confusion Matrix'),
    ('PR_curve.png',          'Precision-Recall Curve'),
]

for fname, title in plots:
    fpath = os.path.join(results_dir, fname)
    if os.path.exists(fpath):
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.imshow(mpimg.imread(fpath))
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()

---
## 🖼️ Step 11 — Visualize Predictions on Test Set

In [ ]:
import glob

test_imgs = list((yolo_out / 'images' / 'test').glob('*.jpg'))
sample    = random.sample(test_imgs, min(12, len(test_imgs)))

best_model.predict(
    source   = [str(p) for p in sample],
    imgsz    = IMG_SIZE,
    conf     = 0.5,
    save     = True,
    project  = './predictions',
    name     = 'test_samples',
    exist_ok = True,
)

pred_imgs = sorted(glob.glob('./predictions/test_samples/*.jpg'))[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('YOLOv8m — Test Set Predictions', fontsize=16, fontweight='bold')
for i, ax in enumerate(axes.flat):
    if i < len(pred_imgs):
        img = cv2.cvtColor(cv2.imread(pred_imgs[i]), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(Path(pred_imgs[i]).stem[:18], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 💾 Step 12 — Export to ONNX

In [ ]:
best_model.export(format='onnx', imgsz=IMG_SIZE, dynamic=False, simplify=True)
onnx_path = BEST_PT.replace('.pt', '.onnx')
print(f'✅ ONNX saved → {onnx_path}')

print('\n' + '='*55)
print('  🎉 Full Pipeline Complete!')
print(f'  Best .pt   : {BEST_PT}')
print(f'  ONNX       : {onnx_path}')
print(f'  YOLO data  : {YOLO_ROOT}')
print('='*55)